# 4.02 Transformer Block：多头注意力、位置编码与前馈网络

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/04_transformer/4.02_transformer_block.ipynb)


## 为什么要学

上一课实现了"注意力运算"。一个完整的 Transformer 层还需要：**多头注意力**、**位置编码**、**前馈网络（FFN）**、**归一化 + 残差**。本课把它们逐个实现，并对照 Gemma（π0 的语言模型和动作专家都基于它）的真实写法：

| 组件 | 原版 Transformer (2017) | Gemma / π0 |
|---|---|---|
| 归一化 | Post-LayerNorm | **Pre-RMSNorm**（π0.5 动作专家：adaRMSNorm） |
| 位置编码 | 正弦（加在输入上） | **RoPE**（旋转 q、k） |
| FFN | ReLU MLP | **GeGLU**：`down(gelu(gate(x)) * up(x))` |
| 注意力头 | 多头 | **多查询注意力（MQA）**：8 个 query 头共享 1 个 key/value 头 |

---

## 1. 多头注意力（Multi-Head Attention）

一个注意力头只能学一种"关注模式"。多头注意力把 $d$ 维特征拆成 $h$ 个头，每个头独立做注意力，最后拼起来：

$$
\text{head}_i=\text{Attention}(XW_Q^{(i)}, XW_K^{(i)}, XW_V^{(i)}),\qquad\text{MHA}(X)=\text{Concat}(\text{head}_1,\dots,\text{head}_h)W_O
$$

实现时不用真的建 $h$ 套矩阵：用一个大的 `Linear(d, h*d_head)` 再 `view` 成 `(B, L, h, d_head)`，然后 `transpose` 成 `(B, h, L, d_head)`，就能并行算所有头。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

### 练习 1：从零实现多头注意力

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dim, n_heads, head_dim=None):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = head_dim or dim // n_heads
        inner = self.n_heads * self.head_dim
        self.q_proj = nn.Linear(dim, inner, bias=False)
        self.k_proj = nn.Linear(dim, inner, bias=False)
        self.v_proj = nn.Linear(dim, inner, bias=False)
        self.o_proj = nn.Linear(inner, dim, bias=False)

    def forward(self, x, mask=None, rope=None):
        """x: (B, L, dim)；mask: bool，可广播到 (B, 1, L, L)；rope: 可选，函数 (q, k) -> (q, k)"""
        B, L, _ = x.shape
        # TODO:
        # 1) q/k/v = proj(x).view(B, L, n_heads, head_dim).transpose(1, 2)   -> (B, h, L, hd)
        # 2) 若 rope 不为 None：q, k = rope(q, k)
        # 3) out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)    -> (B, h, L, hd)
        # 4) out.transpose(1, 2).reshape(B, L, h*hd)，再过 o_proj
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class MultiHeadAttention(nn.Module):
    def __init__(self, dim, n_heads, head_dim=None):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = head_dim or dim // n_heads
        inner = self.n_heads * self.head_dim
        self.q_proj = nn.Linear(dim, inner, bias=False)
        self.k_proj = nn.Linear(dim, inner, bias=False)
        self.v_proj = nn.Linear(dim, inner, bias=False)
        self.o_proj = nn.Linear(inner, dim, bias=False)

    def forward(self, x, mask=None, rope=None):
        """x: (B, L, dim)；mask: bool，可广播到 (B, 1, L, L)；rope: 可选，函数 (q, k) -> (q, k)"""
        B, L, _ = x.shape
        split = lambda t: t.view(B, L, self.n_heads, self.head_dim).transpose(1, 2)
        q, k, v = split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x))
        if rope is not None:
            q, k = rope(q, k)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, L, -1))

In [ ]:
# ✅ 自测：拷贝权重到 nn.MultiheadAttention，结果应一致
torch.manual_seed(0)
dim, h = 32, 4
mine = MultiHeadAttention(dim, h)
ref = nn.MultiheadAttention(dim, h, bias=False, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(torch.cat([mine.q_proj.weight, mine.k_proj.weight, mine.v_proj.weight]))
    ref.out_proj.weight.copy_(mine.o_proj.weight)
x = torch.randn(2, 6, dim)
causal = torch.tril(torch.ones(6, 6, dtype=torch.bool))
out_ref, _ = ref(x, x, x, attn_mask=~causal)          # 注意：nn.MultiheadAttention 的 bool 掩码 True = "禁止"，与我们相反！
assert torch.allclose(mine(x, causal), out_ref, atol=1e-5)
print("通过 ✅ —— 读别人代码时一定要确认掩码的 True 是'允许'还是'禁止'")

> RLinf 的 `RLTSelfAttentionLayer` 用的就是 `nn.MultiheadAttention`，所以它的 `_key_padding_mask` 函数会把 `mask` 取反（`~mask`）。掩码语义的不一致是跨代码库移植时最常见的 bug 之一。

### 多查询注意力（MQA）/ 分组查询注意力（GQA）

Gemma 2B 有 8 个 query 头，但只有 **1** 个 key/value 头（`num_kv_heads=1`），所有 query 头共享同一份 K、V。好处：KV cache 小 8 倍，推理快。实现上只要把 K、V `repeat` 到 8 个头（openpi 的 `repeat_kv`）。

---

## 2. 位置编码

注意力本身对 token 的顺序不敏感：把输入打乱，输出也只是跟着打乱（置换等变）。必须显式告诉模型"谁在第几个位置"。

### (1) 正弦位置编码（原版 Transformer）

$$
PE_{(p, 2i)}=\sin\left(\frac{p}{10000^{2i/d}}\right),\qquad PE_{(p,2i+1)}=\cos\left(\frac{p}{10000^{2i/d}}\right)
$$

不同维度对应不同频率：低维变化快（区分相邻位置），高维变化慢（区分远距离位置）。**π0 用同样的思想编码流匹配的时间步 $t$**（`create_sinusoidal_pos_embedding`，周期范围 `[4e-3, 4.0]`）；RLT 的 `sinusoidal_pe_init` 用它初始化可学习的位置编码。

### 练习 2：正弦位置编码

In [ ]:
def sinusoidal_pe(seq_len, dim):
    """返回 (seq_len, dim)，偶数列 sin、奇数列 cos（与 RLinf 的 sinusoidal_pe_init 相同）"""
    # TODO: position (L,1)；div_term = exp(arange(0, dim, 2) * (-log(10000)/dim))；pe[:, 0::2] = sin(pos*div)，pe[:, 1::2] = cos(pos*div)
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def sinusoidal_pe(seq_len, dim):
    """返回 (seq_len, dim)，偶数列 sin、奇数列 cos（与 RLinf 的 sinusoidal_pe_init 相同）"""
    position = torch.arange(seq_len, dtype=torch.float32)[:, None]
    div_term = torch.exp(torch.arange(0, dim, 2, dtype=torch.float32) * (-math.log(10000.0) / dim))
    pe = torch.zeros(seq_len, dim)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

In [ ]:
# ✅ 自测
pe = sinusoidal_pe(50, 64)
assert pe.shape == (50, 64) and torch.allclose(pe[0, 1::2], torch.ones(32)) and torch.allclose(pe[0, 0::2], torch.zeros(32))
sim = pe @ pe.T
plt.figure(figsize=(9, 3))
plt.subplot(1, 2, 1); plt.imshow(pe.T, aspect="auto", cmap="RdBu"); plt.xlabel("position"); plt.ylabel("dim"); plt.title("sinusoidal PE")
plt.subplot(1, 2, 2); plt.plot(sim[25]); plt.title("PE(25) · PE(p): nearby positions are similar"); plt.xlabel("p")
plt.tight_layout(); plt.show()
print("通过 ✅")

### (2) 旋转位置编码（RoPE）：Gemma / LLaMA 的选择

RoPE 不把位置"加"到输入上，而是在算注意力之前，把 query 和 key 按位置**旋转**一个角度。把每个 $d$ 维向量看成 $d/2$ 个二维向量，第 $i$ 对以频率 $\theta_i=10000^{-2i/d}$ 旋转 $p\theta_i$ 角度。

关键性质：旋转后的点积**只依赖相对位置** $m-n$：

$$
\langle R_m\mathbf q, R_n\mathbf k\rangle=\langle\mathbf q, R_{n-m}\mathbf k\rangle
$$

Gemma（HuggingFace 实现，也就是 openpi 用的）把向量的**前一半和后一半**配对旋转：

$$
\text{RoPE}(\mathbf x)=\mathbf x\odot\cos(p\boldsymbol\theta)+\text{rotate\_half}(\mathbf x)\odot\sin(p\boldsymbol\theta),\qquad\text{rotate\_half}([\mathbf x_1,\mathbf x_2])=[-\mathbf x_2,\mathbf x_1]
$$

在 openpi 里，位置 id 由 `position_ids = cumsum(pad_masks) - 1` 得到。**padding 不占位置号**，所以不同长度的文字后面紧跟的动作 token 位置号是连续的。

### 练习 3：实现 RoPE

In [ ]:
def rope_cos_sin(positions, head_dim, base=10000.0):
    """positions: (B, L) 整数 -> cos, sin: (B, L, head_dim)"""
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))   # (hd/2,)
    freqs = positions[..., None].float() * inv_freq                                              # (B, L, hd/2)
    emb = torch.cat([freqs, freqs], dim=-1)                                                      # (B, L, hd)
    return emb.cos(), emb.sin()

def rotate_half(x):
    # TODO: 前一半 x1、后一半 x2，返回 cat([-x2, x1])
    raise NotImplementedError

def apply_rope(q, k, cos, sin):
    """q, k: (B, h, L, hd)；cos, sin: (B, L, hd)"""
    # TODO: cos/sin 在 head 维上 unsqueeze(1)，然后 q*cos + rotate_half(q)*sin，k 同理
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def rope_cos_sin(positions, head_dim, base=10000.0):
    """positions: (B, L) 整数 -> cos, sin: (B, L, head_dim)"""
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
    freqs = positions[..., None].float() * inv_freq
    emb = torch.cat([freqs, freqs], dim=-1)
    return emb.cos(), emb.sin()

def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, cos, sin):
    """q, k: (B, h, L, hd)；cos, sin: (B, L, hd)"""
    cos, sin = cos[:, None], sin[:, None]
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin

In [ ]:
# ✅ 自测：旋转后的点积只与相对位置有关
torch.manual_seed(0)
q = torch.randn(1, 1, 1, 64); k = torch.randn(1, 1, 1, 64)
def score(m, n):
    cq, sq = rope_cos_sin(torch.tensor([[m]]), 64); ck, sk = rope_cos_sin(torch.tensor([[n]]), 64)
    qr, _ = apply_rope(q, q, cq, sq); kr, _ = apply_rope(k, k, ck, sk)
    return (qr * kr).sum().item()
assert abs(score(3, 7) - score(13, 17)) < 1e-4 and abs(score(3, 7) - score(3, 9)) > 1e-3
assert torch.allclose(apply_rope(q, k, *rope_cos_sin(torch.tensor([[5]]), 64))[0].norm(), q.norm())   # 旋转不改变长度
print("通过 ✅ score(3,7) == score(13,17) =", round(score(3, 7), 4))

---

## 3. 前馈网络：从 MLP 到 GeGLU

每个 token **独立地**过一个两层 MLP（"逐位置"的前馈网络）。注意力负责 token 之间交换信息，FFN 负责对每个 token 自身做非线性变换。Gemma 用**门控**版本 GeGLU：

$$
\text{GeGLU}(\mathbf x)=W_{down}\big(\text{GELU}(W_{gate}\mathbf x)\odot W_{up}\mathbf x\big)
$$

一路做"门"，一路做"值"，逐元素相乘。实践中比普通 MLP 效果更好。

In [ ]:
class GeGLU(nn.Module):
    """Gemma 的 MLP：down(gelu(gate(x)) * up(x))。openpi 中 hidden_activation='gelu_pytorch_tanh'"""

    def __init__(self, dim, hidden):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden, bias=False)
        self.up_proj = nn.Linear(dim, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, dim, bias=False)

    def forward(self, x):
        return self.down_proj(F.gelu(self.gate_proj(x), approximate="tanh") * self.up_proj(x))

---

## 4. 组装：Pre-Norm Transformer Block

$$
\mathbf x\leftarrow\mathbf x+\text{Attn}(\text{Norm}(\mathbf x)),\qquad\mathbf x\leftarrow\mathbf x+\text{FFN}(\text{Norm}(\mathbf x))
$$

Pre-Norm（先归一化再进子层，残差路径上没有归一化）让深层网络训练更稳定，是目前所有大模型的标准结构。

### 练习 4：组装一个 Gemma 风格的 Block

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps, self.weight = eps, nn.Parameter(torch.zeros(dim))   # Gemma 风格：(1 + weight)，weight 初始化为 0

    def forward(self, x):
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * (1.0 + self.weight)


class Block(nn.Module):
    def __init__(self, dim, n_heads, head_dim, mlp_hidden):
        super().__init__()
        self.input_layernorm = RMSNorm(dim)
        self.self_attn = MultiHeadAttention(dim, n_heads, head_dim)
        self.post_attention_layernorm = RMSNorm(dim)
        self.mlp = GeGLU(dim, mlp_hidden)

    def forward(self, x, mask=None, rope=None):
        # TODO: 两个 pre-norm 残差子层
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps, self.weight = eps, nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * (1.0 + self.weight)


class Block(nn.Module):
    def __init__(self, dim, n_heads, head_dim, mlp_hidden):
        super().__init__()
        self.input_layernorm = RMSNorm(dim)
        self.self_attn = MultiHeadAttention(dim, n_heads, head_dim)
        self.post_attention_layernorm = RMSNorm(dim)
        self.mlp = GeGLU(dim, mlp_hidden)

    def forward(self, x, mask=None, rope=None):
        x = x + self.self_attn(self.input_layernorm(x), mask, rope)
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x

In [ ]:
# ✅ 自测：形状、因果性
torch.manual_seed(0)
blk = Block(dim=64, n_heads=4, head_dim=16, mlp_hidden=256)
x = torch.randn(2, 10, 64)
pos = torch.arange(10)[None].expand(2, -1)
cos, sin = rope_cos_sin(pos, 16)
rope = lambda q, k: apply_rope(q, k, cos, sin)
causal = torch.tril(torch.ones(10, 10, dtype=torch.bool))
y = blk(x, causal, rope)
x2 = x.clone(); x2[:, 7:] = torch.randn(2, 3, 64)            # 改变后面的 token
y2 = blk(x2, causal, rope)
assert y.shape == x.shape and torch.allclose(y[:, :7], y2[:, :7], atol=1e-5)   # 因果：前 7 个不受影响
print("通过 ✅ 参数量:", sum(p.numel() for p in blk.parameters()))

### 算一算：π0 的参数量从哪来？

In [ ]:
def gemma_layer_params(width, mlp_dim, n_heads, n_kv_heads, head_dim):
    attn = width * n_heads * head_dim + 2 * width * n_kv_heads * head_dim + n_heads * head_dim * width   # q, k, v, o
    mlp = 3 * width * mlp_dim                                                                            # gate, up, down
    return attn + mlp + 2 * width                                                                        # + 两个 RMSNorm

vlm = 18 * gemma_layer_params(2048, 16384, 8, 1, 256)
expert = 18 * gemma_layer_params(1024, 4096, 8, 1, 256)
print(f"Gemma-2B 18 层（不含词表）≈ {vlm / 1e9:.2f} B   动作专家 gemma_300m ≈ {expert / 1e6:.0f} M")
print("PaliGemma 词表 257152 × 2048 ≈", round(257152 * 2048 / 1e6), "M；SigLIP So400m 视觉塔 ≈ 400 M")

> 注意两个专家的 `n_heads=8`、`head_dim=256` **完全相同**，只有宽度（width）不同。这正是它们能在每一层**拼接 q/k/v、共享一次注意力**的前提（7.02 课详解）。

---

## 小结

- 多头注意力：拆头 → 并行注意力 → 拼接 → 输出投影；MQA 让多个 query 头共享 K/V。
- 位置编码：正弦（加在输入上，也用于编码流匹配时间 t）；RoPE（旋转 q、k，点积只依赖相对位置）。
- Gemma 风格 Block：Pre-RMSNorm + RoPE 注意力 + GeGLU，残差连接。
- 读代码时注意掩码语义：PyTorch `nn.MultiheadAttention` 的 bool 掩码 True 表示"禁止"，`F.scaled_dot_product_attention` 和 openpi 的 True 表示"允许"。